Sample:
preferences = [
    "Funny mystery",
    "No horror",
    "Under 2 hours",
    "Like Interstellar"
]

In [14]:
# pip install sentence-transformers

In [15]:
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("all-MiniLM-L6-v2")
# text="A movie with me"
# embedding = model.encode(text)
# print(type(embedding))
# print(embedding)

# sentence1 = "A team of astronauts travel through space."
# sentence2 = "People explore the universe."
# sentence3 = "A romantic love story."
# from sentence_transformers import util
# embedding1 = model.encode(sentence1)
# embedding2 = model.encode(sentence2)
# embedding3 = model.encode(sentence3)
# score12 = util.cos_sim(embedding1, embedding2)
# score13 = util.cos_sim(embedding1, embedding3)
# print(score12)
# print(score13)

In [16]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv("TMDB_movie_dataset.csv")
df=df.head(10000)
df.head()

KeyboardInterrupt: 

In [ ]:
df.columns.tolist()

['id',
 'title',
 'vote_average',
 'vote_count',
 'status',
 'release_date',
 'revenue',
 'runtime',
 'adult',
 'backdrop_path',
 'budget',
 'homepage',
 'imdb_id',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'poster_path',
 'tagline',
 'genres',
 'production_companies',
 'production_countries',
 'spoken_languages',
 'keywords']

In [ ]:
#Not required features-> adult,status,revenue,budget,original_language,original_title,poster_path,production_companies,production_countries,spoken_languages
#Important->title,release_date,runtime,overview,popularity,tagline,genres,keywords

df = df.drop(columns=["backdrop_path","adult","status","revenue","budget","homepage","original_language","original_title","poster_path","production_companies","production_countries","spoken_languages"])
df.columns.to_list()

['id',
 'title',
 'vote_average',
 'vote_count',
 'release_date',
 'runtime',
 'imdb_id',
 'overview',
 'popularity',
 'tagline',
 'genres',
 'keywords']

In [ ]:
cols = ["title", "overview", "tagline", "genres", "keywords"]
df[cols] = df[cols].fillna("")
df = df.dropna(subset=["title"])

df["movie_text"] = (
    df["title"] + " " +
    df["overview"] + " " +
    df["tagline"] + " " +
    df["genres"] + " " +
    df["keywords"]
)

print(df["movie_text"].iloc[0])

## Semantic Search

In [ ]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")

movie_texts = df["movie_text"].tolist()
embeddings = model.encode(
    movie_texts,
    batch_size=64,
    show_progress_bar=True
)
print(embeddings.shape)
np.save("movie_embeddings.npy", embeddings)
# The order of the embeddings is the same as the order of the DataFrame.
    

c:\Users\dhruv\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 126/126 [02:31<00:00,  1.20s/it]


(8013, 384)


In [ ]:
from sentence_transformers import util
import torch

user_input = input("Enter your preference: ")

user_embedding = model.encode(user_input, convert_to_tensor=True)

movie_embeddings = torch.tensor(embeddings)

similarities = util.cos_sim(user_embedding, movie_embeddings)[0]

top_indices = similarities.argsort(descending=True)
 
for idx in top_indices[:10]:
    idx = idx.item()
    print(f"{similarities[idx]:.4f} - {df.iloc[idx]['title']}")

0.5319 - Interstellar
0.5021 - High Life
0.5014 - Close Encounters of the Third Kind
0.4940 - Cinema Paradiso
0.4843 - Starman
0.4683 - Star Wars: The Force Awakens
0.4681 - Fanboys
0.4666 - Room 237
0.4664 - The Adventures of Buckaroo Banzai Across the 8th Dimension
0.4594 - Mulholland Drive


## Search for similar movie

In [ ]:
movie_title_input = input("Enter movie title").lower()

title_to_index = {t.lower(): i for i, t in zip(df.index, df["title"])}

if movie_title_input not in title_to_index:
    print(f"'{movie_title_input}' not found in the dataset. Check spelling or try another title.")
else:
    match_idx = title_to_index[movie_title_input]
    search_movie_embedding = embeddings[df.index.get_loc(match_idx)]

    similar_movies = util.cos_sim(search_movie_embedding, embeddings)[0]
    top_indices = similar_movies.argsort(descending=True)
    count = 0
    for idx in top_indices:
        idx = idx.item()
        if df.iloc[idx]["title"].lower() == movie_title_input:
            continue
        print(f"{similar_movies[idx]:.4f} - {df.iloc[idx]['title']}")
        count += 1
        if count == 10:
            break

## Weighted rating (accounts for vote count, not just vote_average)

In [ ]:
C = df["vote_average"].mean()
m = df["vote_count"].quantile(0.90)  # only movies above this vote_count are trusted at near-full weight

def weighted_rating(row, m=m, C=C):
    v = row["vote_count"]
    R = row["vote_average"]
    return (v / (v + m)) * R + (m / (v + m)) * C

df["weighted_rating"] = df.apply(weighted_rating, axis=1)

# normalize weighted_rating to 0-1 so it's on the same scale as cosine similarity
df["weighted_rating_norm"] = (df["weighted_rating"] - df["weighted_rating"].min()) / (df["weighted_rating"].max() - df["weighted_rating"].min())

df[["title", "vote_average", "vote_count", "weighted_rating"]].sort_values("weighted_rating", ascending=False).head(10)

`alpha` controls how much semantic match matters vs. how well-regarded the movie is.
`alpha=1.0` -> pure semantic search (ignores rating). `alpha=0.7` is a reasonable starting point - prioritizes relevance but still nudges well-loved movies above obscure/poorly-rated ones with similar text.

In [ ]:
alpha = 0.7  # weight on semantic similarity; (1 - alpha) goes to weighted_rating

user_input = input("Enter your preference: ")

user_embedding = model.encode(user_input, convert_to_tensor=True)
movie_embeddings = torch.tensor(embeddings)

similarities = util.cos_sim(user_embedding, movie_embeddings)[0]

final_scores = alpha * similarities.numpy() + (1 - alpha) * df["weighted_rating_norm"].values

top_indices = final_scores.argsort()[::-1]

for idx in top_indices[:10]:
    print(f"{final_scores[idx]:.4f} (sim={similarities[idx]:.4f}, rating={df.iloc[idx]['weighted_rating']:.2f}) - {df.iloc[idx]['title']}")